In [ ]:
from pathlib import Path
import re
import logging

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions

logging.basicConfig(level=logging.WARNING)

# ── Paths ─────────────────────────────────────────────────────────────────────
DOCS_DIR   = Path("documents")
OUTPUT_DIR = Path("output")

# All PDFs to process — edit this list or use glob to pick a subset
PDF_FILES = sorted(DOCS_DIR.glob("*.pdf"))

print(f"Found {len(PDF_FILES)} PDF(s):")
for p in PDF_FILES:
    print(f"  {p.name}")

In [ ]:
# ── Docling converter setup ────────────────────────────────────────────────────
#
# generate_picture_images=True   → populates pic.image.pil_image for every figure
# do_formula_enrichment=True     → runs CodeFormulaV2 (downloads ~1 GB on first run)
#                                  to convert detected formula regions to LaTeX;
#                                  set False to skip formula parsing
# images_scale=2.0               → render pages at 2× resolution for sharper image crops

FORMULA_ENRICHMENT = True   # set False if CodeFormulaV2 model is not available

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2.0
pipeline_options.do_formula_enrichment = FORMULA_ENRICHMENT

converter = DocumentConverter(
    format_options={"pdf": PdfFormatOption(pipeline_options=pipeline_options)}
)

print("Converter ready.")
print(f"  formula enrichment: {FORMULA_ENRICHMENT}")

In [ ]:
# ── Helper: save pictures to disk ─────────────────────────────────────────────

def save_pictures(doc, img_dir: Path) -> list[Path | None]:
    """
    Save every PictureItem image from a DoclingDocument to `img_dir`.
    Returns an ordered list of saved paths (None for pictures with no image data).
    """
    img_dir.mkdir(parents=True, exist_ok=True)
    paths: list[Path | None] = []

    for i, pic in enumerate(doc.pictures):
        if pic.image is None:
            paths.append(None)
            continue

        pil_img = pic.image.pil_image
        # Skip tiny images (likely decorative lines / separators)
        if pil_img.width < 40 or pil_img.height < 40:
            paths.append(None)
            continue

        path = img_dir / f"img_{i:03d}.png"
        pil_img.save(path, format="PNG")
        paths.append(path)

    saved = sum(1 for p in paths if p is not None)
    print(f"    Saved {saved}/{len(paths)} images → {img_dir}")
    return paths


# ── Helper: build markdown with real image references ─────────────────────────

def build_markdown(
    doc,
    img_paths: list[Path | None],
    md_path: Path,
) -> str:
    """
    Export docling document to markdown, replacing every `<!-- image -->` placeholder
    with a proper `![fig_N](relative/path.png)` reference.

    Formula items (when do_formula_enrichment=True) are automatically exported as
    LaTeX by docling's export_to_markdown.

    `md_path` is used to compute relative image paths.
    """
    PLACEHOLDER = "<!-- image -->"

    raw_md = doc.export_to_markdown(
        image_mode="placeholder",
        image_placeholder=PLACEHOLDER,
        page_break_placeholder="\n\n---\n\n",
        escape_html=False,        # keep & < > readable in the output
        escape_underscores=False, # avoid \_word\_ noise in headings
    )

    placeholder_count = raw_md.count(PLACEHOLDER)
    if placeholder_count != len(img_paths):
        print(
            f"    [WARN] placeholder count ({placeholder_count}) != "
            f"picture count ({len(img_paths)}) — some images may be misaligned"
        )

    parts = raw_md.split(PLACEHOLDER)
    result: list[str] = []
    img_idx = 0

    for i, part in enumerate(parts):
        result.append(part)
        if i < len(parts) - 1:  # not the last segment
            if img_idx < len(img_paths) and img_paths[img_idx] is not None:
                rel = img_paths[img_idx].relative_to(md_path.parent)
                result.append(f"![fig_{img_idx}]({rel})")
            else:
                result.append(PLACEHOLDER)  # preserve if no image saved
            img_idx += 1

    return "".join(result)


print("Helpers defined.")

In [ ]:
# ── Main pipeline ──────────────────────────────────────────────────────────────

def process_pdf(pdf_path: Path, output_dir: Path, converter) -> Path:
    """
    Convert a single PDF to a self-contained markdown file.

    Output layout:
        output_dir/<stem>/
            <stem>.md          ← markdown with image references and LaTeX formulas
            images/
                img_000.png
                img_001.png
                ...
    """
    stem = pdf_path.stem
    doc_dir = output_dir / stem
    img_dir = doc_dir / "images"
    md_path = doc_dir / f"{stem}.md"

    doc_dir.mkdir(parents=True, exist_ok=True)

    print(f"[{stem}] Converting...")
    result = converter.convert(pdf_path)
    doc = result.document
    print(f"    {len(doc.pictures)} pictures, {len(doc.tables)} tables")

    # Save images
    img_paths = save_pictures(doc, img_dir)

    # Build and save markdown
    md = build_markdown(doc, img_paths, md_path)
    md_path.write_text(md, encoding="utf-8")

    print(f"    Written: {md_path}  ({len(md):,} chars)")
    return md_path


print("Pipeline function defined.")

In [ ]:
# ── Run on all PDFs ────────────────────────────────────────────────────────────

output_paths = []
for pdf_path in PDF_FILES:
    out = process_pdf(pdf_path, output_dir=OUTPUT_DIR / "markdown", converter=converter)
    output_paths.append(out)

print(f"\nDone. {len(output_paths)} file(s) written.")
for p in output_paths:
    print(f"  {p}")

In [ ]:
# ── Quick preview of one output ───────────────────────────────────────────────

if output_paths:
    md_text = output_paths[0].read_text(encoding="utf-8")
    print(f"Preview: {output_paths[0].name}  ({len(md_text):,} chars)")
    print("─" * 60)
    print(md_text[:3000])
    print("─" * 60)
    # Report image refs and formula blocks found
    img_refs = re.findall(r'!\[fig_\d+\]\([^)]+\)', md_text)
    formula_blocks = re.findall(r'\$\$[^$]+\$\$', md_text, re.DOTALL)
    inline_formulas = re.findall(r'(?<!\$)\$(?!\$)[^$\n]+\$(?!\$)', md_text)
    print(f"\nImage references : {len(img_refs)}")
    print(f"Display formulas : {len(formula_blocks)}")
    print(f"Inline formulas  : {len(inline_formulas)}")
    if formula_blocks:
        print("\nFirst display formula:")
        print(formula_blocks[0][:300])